# Barnacle instance segmentation ? Colab T4

Select **Runtime ? Change runtime type ? T4 GPU**, then run cells in order.
This notebook fine-tunes **Mask R-CNN R50-FPN + PointRend**. Upload the locally prepared `barnacle_colab.zip` when prompted.

The current labels are **provisional**. The first run is an exploratory, single-tile training diagnostic, not validated scientific inference. The later fine-tuning cell is off by default.


In [ ]:
import sys, subprocess
import torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__, "CUDA runtime:", torch.version.cuda)
assert torch.cuda.is_available(), "Select a T4 GPU runtime, then reconnect."
print("GPU:", torch.cuda.get_device_name())
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
subprocess.run(["nvidia-smi"], check=True)


## Upload the training bundle

The ZIP includes prepared image/label pairs, review artifacts, validation code, and the training scripts. It excludes your local virtual environment. Extract into a new directory; existing work is preserved.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, json, os
uploaded = files.upload()
archives = [Path(name) for name in uploaded if name.lower().endswith(".zip")]
assert len(archives) == 1, "Upload exactly one barnacle training ZIP."
EXTRACT_ROOT = Path("/content/barnacle_workspace")
assert not EXTRACT_ROOT.exists() or not any(EXTRACT_ROOT.iterdir()), "Choose a new EXTRACT_ROOT to preserve existing files."
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archives[0]) as archive:
    for member in archive.infolist():
        destination = (EXTRACT_ROOT / member.filename).resolve()
        assert destination.is_relative_to(EXTRACT_ROOT.resolve()), "Unsafe archive path"
        assert (member.external_attr >> 16) & 0o170000 != 0o120000, "Symlinks are not supported"
    archive.extractall(EXTRACT_ROOT)
WORKSPACE = EXTRACT_ROOT / "barnacle_training"
DATASET = WORKSPACE / "dataset"
assert (DATASET / "dataset_summary.json").is_file()
print("Workspace:", WORKSPACE)
summary = json.loads((DATASET / "dataset_summary.json").read_text())
print("Reviewed training data:", summary["training_ready"])
for scene in summary["scenes"]:
    print(scene["id"], scene["split"], "tiles:", scene["tile_count"], "label count:", scene["candidate_instance_count"])
del uploaded


## Validate the uploaded data

Structural validation checks mask pixels, COCO geometry, tile coordinates, and split separation. It cannot establish biological correctness. The existing annotations need review; 55 ambiguous components were flagged in the initial build.


In [ ]:
subprocess.run([sys.executable, str(WORKSPACE / "training/train_pointrend.py"),
                "--dataset", str(DATASET), "--inspect-data"], check=True)
from IPython.display import display, Image
# This is a visualization of SUPPLIED LABELS, not model predictions.
display(Image(filename=str(DATASET / "scenes/img2/annotation_preview.jpg"), width=700))


## Install Detectron2 and PointRend

This compiles Detectron2 for the runtime GPU against Colab's existing PyTorch. Compilation can take several minutes. The setup checks CUDA and torchvision operators before building. Colab images change; if this cell fails, retain the error log rather than installing random CUDA/PyTorch versions.

The source commit and environment are recorded with each run. To reproduce a previous environment, replace `main` with its saved Detectron2 commit hash. The local Windows preparation environment is unaffected.


In [ ]:
DETECTRON2_ROOT = Path("/content/detectron2")
DETECTRON2_REVISION = "main"
subprocess.run([sys.executable, str(WORKSPACE / "training/colab_setup.py"),
                "--checkout", str(DETECTRON2_ROOT), "--revision", DETECTRON2_REVISION], check=True)


## Choose persistent checkpoint storage

Google Drive is recommended so a disconnected runtime does not erase checkpoints. This cell asks you to connect Drive. Set `SAVE_TO_DRIVE=False` to keep everything in `/content`, then download checkpoints before ending the runtime.


In [ ]:
from datetime import datetime, timezone
SAVE_TO_DRIVE = True
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS_ROOT = Path("/content/drive/MyDrive/BarnacleChallenge/runs")
else:
    RUNS_ROOT = Path("/content/barnacle_runs")
RUN_ROOT = RUNS_ROOT / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
RUN_ROOT.mkdir(parents=True, exist_ok=False)
print("Run outputs:", RUN_ROOT)


## First diagnostic: 50 steps on one tile

`ALLOW_PROVISIONAL=True` explicitly permits exploratory training with the current labels. Unresolved annotation areas are **not automatically ignored** by the generic trainer, so their missing/ambiguous labels can affect learning and metrics. Review/correct the dataset before treating results as scientific evidence. Set this to False for reviewed-data enforcement.

Smoke mode repeats one training tile, disables augmentation, and skips quantitative validation. It checks model loading, masks, gradients and checkpoint writing. Fifty steps are not a promise of convergence or good predictions. Warnings about COCO classification/mask output layers being reinitialized are expected when switching from 80 classes to one.


In [ ]:
sys.path.insert(0, str(WORKSPACE / "training"))
from live_colab import run_with_live_plot

ALLOW_PROVISIONAL = True  # Explicit diagnostic opt-in; current data is not reviewed.
SMOKE_STEPS = 50
SMOKE_OUTPUT = RUN_ROOT / "smoke"
common = [sys.executable, str(WORKSPACE / "training/train_pointrend.py"),
          "--dataset", str(DATASET), "--detectron2-root", str(DETECTRON2_ROOT)]
if ALLOW_PROVISIONAL:
    common += ["--allow-provisional"]
run_with_live_plot(common + ["--mode", "smoke", "--steps", str(SMOKE_STEPS),
                            "--output", str(SMOKE_OUTPUT)], SMOKE_OUTPUT)
LATEST_OUTPUT = SMOKE_OUTPUT


## Inspect actual predictions and loss

Left: supplied labels. Right: actual predictions from the trained diagnostic model, displayed at score ? 0.5. A blank right panel can mean no prediction passed that threshold; it does not necessarily mean the code failed. Check loss, then extend the diagnostic or review labels/settings. A falling loss on the repeated tile demonstrates fitting, not generalization.


In [ ]:
import matplotlib.pyplot as plt
metrics = [json.loads(line) for line in (SMOKE_OUTPUT / "metrics.json").read_text().splitlines() if line.strip()]
loss_rows = [row for row in metrics if "total_loss" in row]
if loss_rows:
    plt.plot([row["iteration"] for row in loss_rows], [row["total_loss"] for row in loss_rows])
    plt.xlabel("Training iteration")
    plt.ylabel("Total training loss")
    plt.grid(alpha=0.3)
    plt.show()
for split in ("train", "val"):
    print(split, "? supplied labels left, model predictions right")
    display(Image(filename=str(SMOKE_OUTPUT / "previews" / f"{split}_comparison.jpg"), width=1000))
print("Checkpoint:", SMOKE_OUTPUT / "model_final.pth")


## Optional: fine-tune on all nine training tiles

Run this only after inspecting the diagnostic. This is a **new run from pretrained weights**, not a continuation of the single-tile diagnostic. It uses flips and modest brightness/contrast changes, evaluates the 16 held-out validation tiles every 100 steps, and saves the best validation mask-AP checkpoint plus the final checkpoint.

600 steps is an initial experiment setting, not an established optimum. Training uses one source photograph, validation another; metrics are exploratory while labels remain provisional. The detection cap is 600, above the observed maximum 396 labels/tile; reported AP uses that custom cap rather than standard COCO's 100. Overlapping tile counts must not be summed into a photograph count.


In [ ]:
RUN_FINETUNE = False  # Change to True when ready to start this separate experiment.
FINETUNE_STEPS = 600
FINETUNE_OUTPUT = RUN_ROOT / "finetune"
if RUN_FINETUNE:
    run_with_live_plot(common + ["--mode", "finetune", "--steps", str(FINETUNE_STEPS),
                                "--output", str(FINETUNE_OUTPUT)], FINETUNE_OUTPUT)
    LATEST_OUTPUT = FINETUNE_OUTPUT
    print((FINETUNE_OUTPUT / "validation_metrics.json").read_text())
    display(Image(filename=str(FINETUNE_OUTPUT / "previews/val_comparison.jpg"), width=1000))
else:
    print("Skipped. Set RUN_FINETUNE=True after inspecting the diagnostic.")


## Keep these artifacts

- `model_final.pth`: final model and training state; `model_best.pth` in fine-tune runs: best validation mask AP.
- `config.yaml`, `run_manifest.json`, `environment.json`: settings, label status, source commit, and runtime versions.
- `metrics.json`: losses and periodic validation metrics; `validation_metrics.json`: final fine-tune metrics.
- `previews/*_comparison.jpg`: clearly separated labels and predictions; `*_predictions.json`: lossless predicted masks and scores in tile coordinates.

If using Drive, outputs are already in the printed run folder. Otherwise download checkpoints from Colab's Files panel before disconnecting. A later tiled-inference/duplicate-reconciliation step is still needed for whole-photo predictions and counts. Ruler calibration and scientific morphometrics are separate subsequent work.

References: [PointRend](https://github.com/facebookresearch/detectron2/tree/main/projects/PointRend), [Detectron2 installation](https://detectron2.readthedocs.io/en/latest/tutorials/install.html).


## Resume an existing run and add steps

Use the directory containing your **200-step checkpoint** below. Resume restores model and optimizer state and retains the original mode (one-tile smoke or all-tile fine-tuning). It saves a new directory, preserving the old run. `ADDITIONAL_STEPS=200` turns 200 completed steps into 400 total steps. The plot uses the absolute step counter.

To move from one-tile smoke to all training tiles, instead use the fine-tuning cell with `--weights` pointing to the 200-step checkpoint. That starts a new optimizer/counter for the new training procedure.


In [ ]:
RUN_RESUME = False  # Change to True and set PREVIOUS_RUN below.
if RUN_RESUME:
    # Run this optional cell only when you want to extend a saved run.
    from pathlib import Path
    import json, sys, torch
    sys.path.insert(0, str(WORKSPACE / "training"))
    from live_colab import run_with_live_plot

    PREVIOUS_RUN = Path("PASTE_THE_FOLDER_CONTAINING_YOUR_200_STEP_CHECKPOINT")
    ADDITIONAL_STEPS = 200
    checkpoint_path = PREVIOUS_RUN / "model_final.pth"
    assert checkpoint_path.is_file(), "Set PREVIOUS_RUN to your existing run folder."
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    completed = int(checkpoint["iteration"]) + 1
    assert ADDITIONAL_STEPS > 0
    target = completed + ADDITIONAL_STEPS
    del checkpoint
    previous = json.loads((PREVIOUS_RUN / "run_manifest.json").read_text())
    RESUMED_OUTPUT = PREVIOUS_RUN.parent / f"{PREVIOUS_RUN.name}_to_{target}"
    command = [sys.executable, str(WORKSPACE / "training/train_pointrend.py"),
               "--dataset", str(DATASET), "--detectron2-root", str(DETECTRON2_ROOT),
               "--mode", previous["mode"], "--resume-from", str(checkpoint_path),
               "--steps", str(target), "--output", str(RESUMED_OUTPUT),
               "--tile-index", str(previous.get("training_tile_index") or 0)]
    if ALLOW_PROVISIONAL:
        command += ["--allow-provisional"]
    print(f"Continue from {completed} to {target}: {ADDITIONAL_STEPS} additional steps")
    run_with_live_plot(command, RESUMED_OUTPUT)
    LATEST_OUTPUT = RESUMED_OUTPUT
    print("Use RUN_OUTPUT = LATEST_OUTPUT when viewing the newest predictions.")
else:
    print("Resume skipped. Set RUN_RESUME=True and choose your checkpoint folder.")


## Unseen-image environment demo and model preservation

Use the same Colab session with the trained checkpoint. Download `barnacle_unseen_demo.zip` from the workspace; it includes the two unseen original photographs and demo code, not another training dataset. Set `RUN_UNSEEN_DEMO=True` below. Choose `TRAINED_RUN=LATEST_OUTPUT` only if that variable points to the exact run you want; an explicit Drive path also works. The selected checkpoint path is printed.

The demo first preserves the chosen full checkpoint/config in a timestamped model snapshot, then predicts on `unseen_img1` (automatic central crop) and `unseen_img2` (entire pre-cropped image). It displays mask-only reports and counts at 0.2/0.3/0.5. No labels exist for these inputs, so AP and scientific accuracy are unknown. Reports and model archives stay beside the trained run (Drive if that run is on Drive).


In [ ]:
RUN_UNSEEN_DEMO = False  # Change to True when ready to test your trained model.
if RUN_UNSEEN_DEMO:
    from google.colab import files
    from pathlib import Path
    import io, zipfile, sys
    uploaded = files.upload()  # Select barnacle_unseen_demo.zip
    name = next(name for name in uploaded if name.endswith(".zip"))
    with zipfile.ZipFile(io.BytesIO(uploaded[name])) as archive:
        for member in archive.infolist():
            relative = Path(member.filename)
            assert relative.parts[0] in {"training", "src", "demo"}
            destination = (WORKSPACE / relative).resolve()
            assert destination.is_relative_to(WORKSPACE.resolve())
            assert (member.external_attr >> 16) & 0o170000 != 0o120000
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(archive.read(member))
    sys.path.insert(0, str(WORKSPACE / "training"))
    from demo_colab import run_unseen_demo
    TRAINED_RUN = LATEST_OUTPUT  # Or an explicit Path to your desired trained run.
    CHECKPOINT_NAME = "model_final.pth"
    SCORE_THRESHOLD = 0.3
    DEMO_ROOT = run_unseen_demo(WORKSPACE, TRAINED_RUN,
                               checkpoint_name=CHECKPOINT_NAME,
                               score_threshold=SCORE_THRESHOLD)
else:
    print("Unseen demo skipped. Set RUN_UNSEEN_DEMO=True after selecting your trained run.")
